In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# Load Dataset
dataset = pd.read_csv("CKD.csv")
dataset

,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,2.000000,76.459948,c,3.0,0.0,normal,abnormal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,yes,no,yes
1,3.000000,76.459948,c,2.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,34.000000,12300.000000,4.705597,no,no,no,yes,poor,no,yes
2,4.000000,76.459948,a,1.0,0.0,normal,normal,notpresent,notpresent,99.000000,...,34.000000,8408.191126,4.705597,no,no,no,yes,poor,no,yes
3,5.000000,76.459948,d,1.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,38.868902,8408.191126,4.705597,no,no,no,yes,poor,yes,yes
4,5.000000,50.000000,c,0.0,0.0,normal,normal,notpresent,notpresent,148.112676,...,36.000000,12400.000000,4.705597,no,no,no,yes,poor,no,yes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,219.000000,...,37.000000,9800.000000,4.400000,no,no,no,yes,poor,no,yes
395,51.492308,70.000000,c,0.0,2.0,normal,normal,notpresent,notpresent,220.000000,...,27.000000,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes
396,51.492308,70.000000,c,3.0,0.0,normal,normal,notpresent,notpresent,110.000000,...,26.000000,9200.000000,3.400000,yes,yes,no,poor,poor,no,yes
397,51.492308,90.000000,a,0.0,0.0,normal,normal,notpresent,notpresent,207.000000,...,38.868902,8408.191126,4.705597,yes,yes,no,yes,poor,yes,yes


In [3]:
# Independent and Dependent Variables
Independent = dataset.drop(columns=['classification'])
Dependent = dataset['classification']

In [4]:
# Convert categorical columns
Independent = pd.get_dummies(Independent, drop_first=True)

In [5]:
# Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(Independent, Dependent, test_size=1/3, random_state=0)

In [6]:
# Feature Scaling
from sklearn.preprocessing import StandardScaler
sc = StandardScaler()
X_train = sc.fit_transform(X_train)
X_test = sc.transform(X_test)

In [7]:
# Grid Search for Naive Bayes
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
param_grid = {'var_smoothing': np.logspace(0, -9, num=100)}

grid = GridSearchCV(
    GaussianNB(),
    param_grid,
    refit=True,
    verbose=3,
    n_jobs=-1,
    scoring='f1_weighted'
)

In [8]:
grid.fit(X_train, y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits


GridSearchCV(estimator=GaussianNB(), n_jobs=-1,
             param_grid={'var_smoothing': array([1.00000000e+00, 8.11130831e-01, 6.57933225e-01, 5.33669923e-01,
       4.32876128e-01, 3.51119173e-01, 2.84803587e-01, 2.31012970e-01,
       1.87381742e-01, 1.51991108e-01, 1.23284674e-01, 1.00000000e-01,
       8.11130831e-02, 6.57933225e-02, 5.33669923e-02, 4.32876128e-02,
       3.51119173e-02, 2.84803587e-02...
       1.23284674e-07, 1.00000000e-07, 8.11130831e-08, 6.57933225e-08,
       5.33669923e-08, 4.32876128e-08, 3.51119173e-08, 2.84803587e-08,
       2.31012970e-08, 1.87381742e-08, 1.51991108e-08, 1.23284674e-08,
       1.00000000e-08, 8.11130831e-09, 6.57933225e-09, 5.33669923e-09,
       4.32876128e-09, 3.51119173e-09, 2.84803587e-09, 2.31012970e-09,
       1.87381742e-09, 1.51991108e-09, 1.23284674e-09, 1.00000000e-09])},
             scoring='f1_weighted', verbose=3)

In [9]:
# Results
re = grid.cv_results_

In [10]:
# Prediction
grid_predictions = grid.predict(X_test)

In [11]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, grid_predictions)

# F1 Score
from sklearn.metrics import f1_score
f1_weighted = f1_score(
    y_test,
    grid_predictions,
    average='weighted'
)

print(
    "The f1_weighted value for best parameter {}:".format(grid.best_params_),
    f1_weighted
)
print("The Confusion Matrix:\n", cm)

The f1_weighted value for best parameter {'var_smoothing': 0.005336699231206307}: 0.9775556904684072
The Confusion Matrix:
 [[51  0]
 [ 3 79]]


In [12]:
# ROC AUC Score
from sklearn.metrics import roc_auc_score
roc_auc = roc_auc_score(y_test,grid.predict_proba(X_test)[:, 1])

print("ROC_AUC Score:", roc_auc)

ROC_AUC Score: 1.0


In [13]:
# Grid Search Results Table
table = pd.DataFrame.from_dict(re)

# Classification Report
from sklearn.metrics import classification_report
clf_report = classification_report(
    y_test,
    grid_predictions
)
print(clf_report)

              precision    recall  f1-score   support

          no       0.94      1.00      0.97        51
         yes       1.00      0.96      0.98        82

    accuracy                           0.98       133
   macro avg       0.97      0.98      0.98       133
weighted avg       0.98      0.98      0.98       133



In [15]:
print("OVERALL REPORT for NavieBayes Classification:\n")
print("Best Parameters:\n\t", grid.best_params_)
print("Best Cross Validation Score:\n\t", grid.best_score_)
print("The f1_weighted value for best parameter:\n\t",f1_weighted)
print("The confusion Matrix:\n",cm)
print("ROC_AUC Score:\n\t", roc_auc)
print("classification_report:\n\t",clf_report)

OVERALL REPORT for NavieBayes Classification:

Best Parameters:
	 {'var_smoothing': 0.005336699231206307}
Best Cross Validation Score:
	 0.9850034030491555
The f1_weighted value for best parameter:
	 0.9775556904684072
The confusion Matrix:
 [[51  0]
 [ 3 79]]
ROC_AUC Score:
	 1.0
classification_report:
	               precision    recall  f1-score   support

          no       0.94      1.00      0.97        51
         yes       1.00      0.96      0.98        82

    accuracy                           0.98       133
   macro avg       0.97      0.98      0.98       133
weighted avg       0.98      0.98      0.98       133

